# Using the geeViz MCP Server with Gemini

This notebook shows how to use the geeViz MCP server **outside** of a coding agent (like Claude Code or Cursor). Instead, we connect to the MCP server programmatically and use **Google Gemini** as the LLM to decide which tools to call.

This teaches the core MCP pattern:
1. **Start** an MCP server (subprocess)
2. **List** available tools and their schemas
3. **Send** a user question to the LLM along with the tool definitions
4. **Execute** whichever tool(s) the LLM chooses
5. **Return** the results to the LLM for a final answer

**Requirements:**
- `pip install geeViz google-genai mcp`
- `GEMINI_API_KEY` environment variable (or in a `.env` file)
- Authenticated Earth Engine session

In [1]:
# Step 1: Imports and configuration
import os, json, asyncio
from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

GEMINI_MODEL = "gemini-3-flash-preview"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in your environment or .env file"

# Path to the geeViz MCP server
SERVER_SCRIPT = os.path.join(os.path.dirname(os.getcwd()), "mcp", "server.py")
print(f"MCP server: {SERVER_SCRIPT}")
print(f"Gemini model: {GEMINI_MODEL}")

MCP server: c:\RCR\geeVizBuilder\geeViz\mcp\server.py
Gemini model: gemini-3-flash-preview


## Step 2: Connect to the MCP Server

The MCP SDK provides `StdioServerParameters` and `ClientSession` to launch and talk to a server over stdio. The server runs as a subprocess — just like how Claude Code or Cursor connect to it.

In [2]:
import subprocess
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Define how to launch the server
server_params = StdioServerParameters(
    command="python",
    args=[SERVER_SCRIPT],
)

# We'll store the session globally so other cells can use it
session = None
_client_ctx = None
_session_ctx = None
_errlog = None

async def connect():
    """Connect to the MCP server and return the session."""
    global session, _client_ctx, _session_ctx, _errlog
    
    # On Windows + Jupyter, stderr doesn't have a real fileno().
    # Redirect MCP server stderr to a log file instead.
    _errlog = open(os.path.join(os.path.dirname(SERVER_SCRIPT), "server_stderr.log"), "w")
    
    _client_ctx = stdio_client(server_params, errlog=_errlog)
    read, write = await _client_ctx.__aenter__()
    
    _session_ctx = ClientSession(read, write)
    session = await _session_ctx.__aenter__()
    
    await session.initialize()
    return session

async def disconnect():
    """Cleanly disconnect from the MCP server."""
    global session, _client_ctx, _session_ctx, _errlog
    if _session_ctx:
        await _session_ctx.__aexit__(None, None, None)
    if _client_ctx:
        await _client_ctx.__aexit__(None, None, None)
    if _errlog:
        _errlog.close()
    session = None

await connect()
print(f"Connected to MCP server")

Connected to MCP server


## Step 3: Discover Available Tools

The MCP protocol's `tools/list` method returns all tools with their names, descriptions, and JSON Schema parameter definitions. This is exactly what gets sent to the LLM.

In [3]:
# List all available tools
tools_result = await session.list_tools()
mcp_tools = tools_result.tools

print(f"Available tools: {len(mcp_tools)}\n")
for tool in mcp_tools:
    print(f"  {tool.name}: {tool.description[:80]}..." if len(tool.description) > 80 else f"  {tool.name}: {tool.description}")

Available tools: 30

  run_code: Execute Python/GEE code in a persistent REPL namespace (like Jupyter).

The name...
  inspect_asset: Get detailed metadata for any GEE asset (Image, ImageCollection, FeatureCollecti...
  get_api_reference: Look up the signature and docstring of a geeViz function or module.

Uses Python...
  search_functions: Search for functions across geeViz modules, or list functions in a specific modu...
  get_example: Read the source code of a geeViz example script.

Args:
    example_name: Name o...
  list_examples: List available geeViz example scripts with descriptions.

Args:
    filter: Opti...
  list_assets: List assets in a GEE folder or collection.

Args:
    folder: Full asset path (e...
  track_tasks: Get status of recent Earth Engine tasks.

Args:
    name_filter: Optional case-i...
  map_control: Control the geeView interactive map.

Args:
    action: Action to perform:
     ...
  save_session: Save the accumulated run_code history to a .py script or .ip

## Step 4: Convert MCP Tools to Gemini Function Declarations

Gemini's function calling API expects tools in a specific format. We convert the MCP tool schemas to Gemini function declarations.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

def mcp_schema_to_gemini(schema):
    """Convert an MCP JSON Schema to a Gemini-compatible schema dict."""
    if not schema:
        return {"type": "OBJECT", "properties": {}}
    
    type_map = {
        "string": "STRING",
        "integer": "INTEGER",
        "number": "NUMBER",
        "boolean": "BOOLEAN",
        "array": "ARRAY",
        "object": "OBJECT",
    }
    
    properties = {}
    for name, prop in schema.get("properties", {}).items():
        prop_type = prop.get("type", "string")
        gemini_prop = {"type": type_map.get(prop_type, "STRING")}
        if "description" in prop:
            gemini_prop["description"] = prop["description"]
        properties[name] = gemini_prop
    
    result = {
        "type": "OBJECT",
        "properties": properties,
    }
    if "required" in schema:
        result["required"] = schema["required"]
    return result


# Build ALL function declarations into a SINGLE Tool object
# (Gemini works better with one Tool containing many functions)
function_decls = []
tool_lookup = {}

for tool in mcp_tools:
    tool_lookup[tool.name] = tool
    function_decls.append(types.FunctionDeclaration(
        name=tool.name,
        description=tool.description,
        parameters=mcp_schema_to_gemini(tool.inputSchema),
    ))

gemini_tools = [types.Tool(function_declarations=function_decls)]
print(f"Converted {len(function_decls)} MCP tools into 1 Gemini Tool object")

Converted 30 MCP tools into 1 Gemini Tool object


## Step 5: Build the System Prompt & Agent Loop

Coding agents like Claude Code and Cursor receive `agent-instructions.md` as context alongside the tool definitions. We replicate that here by loading the same file and including it in Gemini's system prompt. This gives Gemini the same domain knowledge about geeViz patterns, dataset pitfalls, function signatures, and tool workflows.

The agent loop then:
1. Sends the user question + system prompt + tool definitions to Gemini
2. If Gemini wants to call a tool, executes it via the MCP session
3. Sends the tool result back to Gemini
4. Repeats until Gemini gives a final text answer (max 6 turns)

In [ ]:
# Load the same agent instructions that coding agents (Claude Code, Cursor) receive.
# This is the ONLY system prompt — no separate instructions needed.
agent_instructions_path = os.path.join(os.path.dirname(SERVER_SCRIPT), "agent-instructions.md")
with open(agent_instructions_path, "r", encoding="utf-8") as f:
    AGENT_INSTRUCTIONS = f.read()

print(f"Loaded agent instructions: {len(AGENT_INSTRUCTIONS):,} chars")

# The agent instructions ARE the system prompt.
# We only append minimal Gemini-specific guardrails (not domain knowledge).
SYSTEM_PROMPT = AGENT_INSTRUCTIONS + "\n\n" + (
    "## Additional rules for this session\n"
    "- Be concise in your final text answer.\n"
    "- Do NOT call the same tool twice with the same arguments.\n"
    "- After getting a useful result, give a final answer promptly.\n"
    "- To show data on the map: use run_code to add layers, then call "
    "map_control(action='view') as a SEPARATE tool call afterward.\n"
)

async def ask_gemini_with_tools(question, max_turns=8):
    """Send a question to Gemini with MCP tools and handle the tool-call loop."""
    
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}\n")
    
    messages = [{"role": "user", "parts": [{"text": question}]}]
    
    for turn in range(max_turns):
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=messages,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                tools=gemini_tools,
                temperature=0.1,
            ),
        )
        
        candidate = response.candidates[0]
        parts = candidate.content.parts
        
        has_function_call = any(hasattr(p, 'function_call') and p.function_call for p in parts)
        
        if not has_function_call:
            text = "".join(p.text for p in parts if hasattr(p, 'text') and p.text)
            print(f"\nGemini's answer:\n{text}")
            return text
        
        # Process function calls
        messages.append({"role": "model", "parts": parts})
        function_responses = []
        
        for part in parts:
            if not (hasattr(part, 'function_call') and part.function_call):
                continue
                
            fc = part.function_call
            tool_name = fc.name
            tool_args = dict(fc.args) if fc.args else {}
            
            # Fix Gemini sending floats for int params
            for k, v in tool_args.items():
                if isinstance(v, float) and v == int(v):
                    tool_args[k] = int(v)
            
            print(f"  [Turn {turn+1}] {tool_name}({json.dumps(tool_args, default=str)[:200]})")
            
            try:
                result = await session.call_tool(tool_name, tool_args)
                tool_result = "\n".join(
                    c.text for c in result.content if hasattr(c, 'text') and c.text
                ) or "(no output)"
                if len(tool_result) > 5000:
                    tool_result = tool_result[:5000] + "\n... (truncated)"
                preview = tool_result[:300].replace('\n', ' ')
                print(f"    -> {preview}{'...' if len(tool_result) > 300 else ''}")
            except Exception as e:
                tool_result = f"Error: {e}"
                print(f"    -> Error: {e}")
            
            function_responses.append(types.Part.from_function_response(
                name=tool_name,
                response={"result": tool_result},
            ))
        
        messages.append({"role": "user", "parts": function_responses})
    
    print("\n(max turns reached)")
    return "(max turns reached)"

print("Agent loop ready.")

---
## Test 1: API Lookup

Ask Gemini to look up a function signature. It should call `get_api_reference`.

In [ ]:
await ask_gemini_with_tools(
    "Look up the signature of getLandsatWrapper in the getImagesLib module. "
    "What are its required parameters?"
)

---
## Test 2: Dataset Discovery

Ask about a dataset. Gemini should use `inspect_asset`.

In [ ]:
await ask_gemini_with_tools(
    "Inspect the GEE asset 'USFS/GTAC/LCMS/v2024-10' and tell me what bands it has."
)

---
## Test 3: Map Visualization

Ask Gemini to add data to the map and open it. Tests `run_code` + `map_control`.

In [ ]:
await ask_gemini_with_tools(
    "Use run_code to add LCMS land cover to the map for Salt Lake County, Utah. "
    "Use sal.getUSCounties for the boundary, Map.addLayer with autoViz, and Map.centerObject. "
    "Then use map_control to open the map."
)

---
## Test 4: Charting

Ask for zonal statistics. Tests `run_code` to set up data, then `extract_and_chart`.

In [ ]:
await ask_gemini_with_tools(
    "First use run_code to create these variables: "
    "lcms_lc = ee.ImageCollection('USFS/GTAC/LCMS/v2024-10').select(['Land_Cover']) "
    "and study_area = ee.Geometry.Point([-111.04, 45.68]).buffer(10000). "
    "Then use extract_and_chart with collection_var='lcms_lc' and geometry_var='study_area'."
)

---
## Test 5: EDW Query

Query the USFS Enterprise Data Warehouse. Tests `search_edw` + `query_edw_features`.

In [ ]:
await ask_gemini_with_tools(
    "Search EDW for 'MTBS', then get the service info for EDW_MTBS_01, "
    "then query layer 75 (2020 Burned Area Boundaries) with "
    "where=\"FIRE_NAME LIKE '%CAMERON PEAK%'\" and out_fields='FIRE_NAME,ACRES,YEAR'. "
    "How many acres did the Cameron Peak fire burn?"
)

---
## Test 6: Geocoding

Geocoding a place name. Tests `geocode`.

In [ ]:
await ask_gemini_with_tools(
    "Geocode 'Yellowstone National Park' and tell me the coordinates and bounding box."
)

---
## Test 7: Open-ended Analysis

A realistic question — no hints about which tools to use. Gemini must figure out the workflow from the agent instructions.

In [ ]:
await ask_gemini_with_tools(
    "What is the current LCMS land cover breakdown near Boise, Idaho? "
    "Show me a bar chart."
)

---
## Test 8: Report Generation

Test the full report workflow: create → add section → generate.

In [ ]:
await ask_gemini_with_tools(
    "Create a report titled 'Vermont Land Cover' with dark theme. "
    "First use run_code to set up: study_area = sal.getUSStates(ee.Geometry.Point([-72.58, 44.26])) "
    "and lcms_img = ee.ImageCollection('USFS/GTAC/LCMS/v2024-10').filter("
    "ee.Filter.calendarRange(2024,2024,'year')).filterBounds(study_area)"
    ".select(['Land_Cover']).mosaic().copyProperties("
    "ee.ImageCollection('USFS/GTAC/LCMS/v2024-10').first()). "
    "Then add a report section with the LCMS image over the study area. "
    "Then generate the report as HTML."
)

---
## Test 9: Example Discovery

Ask Gemini to find and show a relevant example script. Tests `list_examples` + `get_example`.

In [ ]:
await ask_gemini_with_tools(
    "List geeViz examples related to change detection, "
    "then show me the source code of the LANDTRENDR example."
)

---
## Test 10: Fully Open-ended

No hints at all. Gemini must use the agent instructions to figure out the entire workflow.

In [ ]:
await ask_gemini_with_tools(
    "How has forest cover changed in the Lolo National Forest in Montana over the last 40 years?"
)

---
## Cleanup

Disconnect from the MCP server when done.

In [ ]:
await disconnect()
print("Disconnected from MCP server.")

---
## How It Works: The MCP Pattern

```
┌─────────┐    question     ┌─────────┐    tool call     ┌────────────┐
│  User   │ ──────────────> │ Gemini  │ ───────────────> │ geeViz MCP │
│         │                 │  (LLM)  │ <─────────────── │   Server   │
│         │ <────────────── │         │    tool result    │            │
│         │   final answer  │         │                   │  (30 tools)│
└─────────┘                 └─────────┘                   └────────────┘
```

**Key points:**
- The MCP server is a **subprocess** communicating over stdio (JSON-RPC)
- The LLM **never** sees your EE credentials — only tool names and results
- Any LLM with function calling (Gemini, Claude, GPT-4, etc.) can be the brain
- The same server works with Claude Code, Cursor, VS Code Copilot, or this notebook
- Tools are the universal primitive — supported by all MCP clients

### To use with other LLMs
Replace the Gemini client with any LLM that supports function calling:
- **Claude API**: `anthropic.Anthropic().messages.create(tools=[...])`
- **OpenAI**: `openai.chat.completions.create(tools=[...])`
- **Local models**: Ollama, vLLM, etc. with function calling support

The MCP connection code (Steps 2-3) stays exactly the same.